In [1]:
import os
import numpy as np
import pandas as pd

INPUT_DIR = "../input/chaii-hindi-and-tamil-question-answering"
test_path = os.path.join(INPUT_DIR, "test.csv")
sample_path = os.path.join(INPUT_DIR, "sample_submission.csv")

test_df = pd.read_csv(test_path)
sample_sub = pd.read_csv(sample_path)

print("test_df shape:", test_df.shape)
print("sample_sub shape:", sample_sub.shape)
print("test_df columns:", list(test_df.columns))
print("sample_sub columns:", list(sample_sub.columns))



test_df shape: (112, 4)
sample_sub shape: (112, 2)
test_df columns: ['id', 'context', 'question', 'language']
sample_sub columns: ['id', 'PredictionString']


In [2]:
# Display a couple of rows for sanity (kept minimal to avoid large outputs)
test_df.head(2)



,id,context,question,language
0,be799d365,एशियन पेंट्स लिमिटेड एक भारतीय बहुराष्ट्रीय कम...,एशियन पेंट्स कंपनी का मुख्यालय कहाँ स्थित है?,hindi
1,26f356026,स्वामी निगमानन्द परमहंस (18 अगस्त 1880 - 29 नव...,स्वामी निगमानन्द परमहंस के तन्त्र गुरु कौन थे?,hindi


In [3]:
# (Placeholder cell to preserve original ordering intent; no-op)
pass



In [4]:
# FIX: transformers.pipeline import triggered a protobuf runtime error in this environment.
# Use the lower-level classes directly and implement QA inference ourselves.
import torch
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device



device(type='cuda')

In [5]:
# FIX: original code referenced a non-existent local model directory.
# Minimal change while preserving core logic (extractive QA): use a multilingual QA model available via HF.
# This runs offline on Kaggle if model is cached; otherwise it will download (no internet may fail).
# Assumption: Kaggle environment allows HF model download or has it cached.
MODEL_NAME = "deepset/xlm-roberta-base-squad2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME).to(device)
model.eval()




tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

2026-05-14 16:44:52.743658: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778777092.752694      55 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778777092.755386      55 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-14 16:44:52.766906: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


...

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

XLMRobertaForQuestionAnswering(
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, 

In [6]:
# Helper: standard extractive QA decoding (argmax over start/end with max_answer_len constraint).
@torch.inference_mode()
def predict_answer(question: str, context: str, max_answer_len: int = 40) -> str:
    if not isinstance(question, str):
        question = "" if pd.isna(question) else str(question)
    if not isinstance(context, str):
        context = "" if pd.isna(context) else str(context)

    enc = tokenizer(
        question,
        context,
        truncation="only_second",
        max_length=384,
        stride=128,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
        return_tensors="pt",
    )

    # For each overflow chunk, compute best span; then take global best by score.
    best_answer = ""
    best_score = -1e18

    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)

    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    start_logits = outputs.start_logits.detach().cpu()
    end_logits = outputs.end_logits.detach().cpu()

    # enc is batch over chunks
    for i in range(input_ids.size(0)):
        offsets = enc["offset_mapping"][i].tolist()
        sequence_ids = enc.sequence_ids(i)

        s_logits = start_logits[i].numpy()
        e_logits = end_logits[i].numpy()

        # Only allow spans in the context part (sequence_id == 1)
        context_token_idxs = [
            idx
            for idx, sid in enumerate(sequence_ids)
            if sid == 1 and offsets[idx] != (0, 0)
        ]
        if not context_token_idxs:
            continue

        # Get top start/end candidates within context tokens
        # (small, deterministic; keeps runtime manageable)
        top_n = 20
        start_candidates = sorted(
            context_token_idxs, key=lambda t: s_logits[t], reverse=True
        )[:top_n]
        end_candidates = sorted(
            context_token_idxs, key=lambda t: e_logits[t], reverse=True
        )[:top_n]

        for s in start_candidates:
            for e in end_candidates:
                if e < s:
                    continue
                if (e - s + 1) > max_answer_len:
                    continue
                score = float(s_logits[s] + e_logits[e])
                if score > best_score:
                    start_char, _ = offsets[s]
                    _, end_char = offsets[e]
                    ans = context[start_char:end_char].strip()
                    if ans == "":
                        continue
                    best_score = score
                    best_answer = ans

    return best_answer if best_answer != "" else ""




In [7]:
# Quick smoke test on first row
row0 = test_df.iloc[0]
print("Q:", row0["question"])
print("Pred:", predict_answer(row0["question"], row0["context"])[:200])



Q: एशियन पेंट्स कंपनी का मुख्यालय कहाँ स्थित है?
Pred: मुंबई, महाराष्ट्र


In [8]:
# Generate predictions for the full test set (7172 rows).
# FIX: ensure length matches test_df exactly.
from tqdm.auto import tqdm

predict_list = []
for q, c in tqdm(
    zip(test_df["question"].tolist(), test_df["context"].tolist()), total=len(test_df)
):
    predict_list.append(predict_answer(q, c))

len(predict_list), len(test_df)



  0%|          | 0/112 [00:00<?, ?it/s]

(112, 112)

In [9]:
# Attach predictions
test_df = test_df.copy()
test_df["predicted_ans"] = predict_list
test_df[["id", "predicted_ans"]].head(3)



,id,predicted_ans
0,be799d365,"मुंबई, महाराष्ट्र"
1,26f356026,श्री मद स्वामी निगमानंद सरस्वती देव
2,57a56c43f,பெருமூளைப் புறணிப்


In [10]:
# (Original notebook had a large commented context example; keep cell as no-op)
pass



In [11]:
# (Placeholder to preserve numbering; no-op)
pass



In [12]:
# Keep function name from original (now already defined above); no-op to preserve cell.
pass



In [13]:
# (Placeholder; no-op)
pass



In [14]:
# Already executed prediction loop in cell 8; keep cell as no-op to preserve original structure.
pass



In [15]:
# Sanity checks and ensure dtype is string (avoids issues in .str ops)
test_df["predicted_ans"] = test_df["predicted_ans"].fillna("").astype(str)
test_df[["id", "predicted_ans"]].tail(3)



,id,predicted_ans
109,3c3513477,एथेन्स ग्रीस
110,934a3199c,उत्तर अमेरिका
111,4f41fec88,थॉमस हकल वेलर और फ्रेडरिक चैपमैन रोबिन्स


In [16]:
# Apply the same regex cleanup as original, but fix pandas regex warnings and ensure column exists.
import re

regList = [
    r"^[0-9]+\.",
    r"^[0-9]\.",
    r"^[0-9][0-9]\.",
    r"^[0-9][0-9][0-9]\.",
    r"^\([0-9]+\)",
    r"^[0-9]+\)",
]

for reg in regList:
    test_df["predicted_ans"] = test_df["predicted_ans"].str.replace(
        reg, " ", regex=True
    )

test_df["predicted_ans"] = test_df["predicted_ans"].str.replace("(", "", regex=False)
test_df["predicted_ans"] = test_df["predicted_ans"].str.replace(")", "", regex=False)



In [17]:
# (Placeholder; no-op)
pass



In [18]:
test_df[["id", "predicted_ans"]].head(5)



,id,predicted_ans
0,be799d365,"मुंबई, महाराष्ट्र"
1,26f356026,श्री मद स्वामी निगमानंद सरस्वती देव
2,57a56c43f,பெருமூளைப் புறணிப்
3,da062fdbb,पितृहन्ता
4,72fc0d5b5,20 अप्रैल 1889


In [19]:
# FIX: Ensure submission format matches sample_submission (id + PredictionString)
final_test = pd.DataFrame(
    {
        "id": test_df["id"].values,
        "PredictionString": test_df["predicted_ans"].values,
    }
)

# Align ordering with sample_submission ids if needed (safe, score-neutral)
final_test = sample_sub[["id"]].merge(final_test, on="id", how="left")
final_test["PredictionString"] = final_test["PredictionString"].fillna("").astype(str)

final_test.head(3), final_test.shape



(          id                     PredictionString
 0  be799d365                    मुंबई, महाराष्ट्र
 1  26f356026  श्री मद स्वामी निगमानंद सरस्वती देव
 2  57a56c43f                   பெருமூளைப் புறணிப்,
 (112, 2))

In [20]:
final_test.tail(3)



,id,PredictionString
109,3c3513477,एथेन्स ग्रीस
110,934a3199c,उत्तर अमेरिका
111,4f41fec88,थॉमस हकल वेलर और फ्रेडरिक चैपमैन रोबिन्स


In [21]:
# Write valid Kaggle submission
out_path = "submission.csv"
final_test.to_csv(out_path, index=False)

print("Wrote:", out_path)
print("Columns:", list(final_test.columns))
print("Rows:", len(final_test))
print(final_test.head(2).to_string(index=False))

Wrote: submission.csv
Columns: ['id', 'PredictionString']
Rows: 112
       id                    PredictionString
be799d365                   मुंबई, महाराष्ट्र
26f356026 श्री मद स्वामी निगमानंद सरस्वती देव
